# 第13回 人工知能とゲーム

目標：値・手・評価回数を分け、ミニマックスとαβの一致を確かめる。既存venvのPythonカーネルを選び、再起動して全セル実行します。同梱Pythonはimportしません。詳細は同梱guide.html。予想→実行→一つ変更→説明の順に進めます。

In [ ]:
"""第13回：ゲーム木とニム。実行: python game_search.py --demo

利得は常にMAXから見た値。探索は学習ではなく、入力を変更しない。
同じ局面への別経路も別節点として数え、キャッシュは使用しない。
"""
import argparse
from copy import deepcopy
import json

TREE_BASE = {"leaves": [3, 5, 2, 9, 0, 4], "heuristics": [0, 1, 4, 2],
             "depth": 2, "reverse": False, "algorithm": "minimax", "player": 1}
NIM_BASE = {"piles": [3, 2], "player": 1, "reverse": False, "algorithm": "minimax"}


def integer(value, low, high, name):
    if type(value) is not int or not low <= value <= high:
        raise ValueError(f"{name}: {low}〜{high}の整数を指定してください")
    return value


def validate_piles(piles):
    if not isinstance(piles, (list, tuple)) or len(piles) != 2:
        raise ValueError("山は2個です")
    for p in piles:
        integer(p, 0, 4, "石の数")
    if sum(piles) > 6:
        raise ValueError("石は合計6個以下です")


def legal_moves(piles):
    validate_piles(piles)
    return [[i, take] for i, n in enumerate(piles) for take in range(1, n + 1)]


def apply_move(piles, action):
    validate_piles(piles)
    if not isinstance(action, (list, tuple)) or len(action) != 2:
        raise ValueError("手は[山番号,取る個数]です")
    i, take = action
    integer(i, 0, 1, "山番号")
    integer(take, 1, 4, "取る個数")
    if take > piles[i]:
        raise ValueError("その山には指定した数の石がありません")
    result = list(piles)
    result[i] -= take
    return result


def _config(config, kind):
    base = TREE_BASE if kind == "tree" else NIM_BASE
    if not isinstance(config, dict) or set(config) - set(base):
        raise ValueError("未知の設定項目です")
    c = deepcopy({**base, **config})
    if type(c["player"]) is not int or c["player"] not in (-1, 1):
        raise ValueError("手番はMAX=1またはMIN=-1です")
    if type(c["reverse"]) is not bool or c["algorithm"] not in ("minimax", "alphabeta"):
        raise ValueError("探索順・方式が不正です")
    if kind == "tree":
        integer(c["depth"], 0, 2, "探索深さ")
        for key, length in (("leaves", 6), ("heuristics", 4)):
            if not isinstance(c[key], (list, tuple)) or len(c[key]) != length:
                raise ValueError(f"{key}は{length}個です")
            for x in c[key]:
                integer(x, -100, 100, key)
    else:
        validate_piles(c["piles"])
    return c


def _search(c, kind):
    """深さ優先探索。履歴中の±無限大はJSONで明示できる文字列にする。"""
    trace, pruned = [], []
    stats = {"visited_nodes": 0, "evaluations": 0, "terminal_evaluations": 0,
             "cutoff_evaluations": 0}
    ab = c["algorithm"] == "alphabeta"

    def bound_number(x):
        return "-inf" if x == -float("inf") else "+inf" if x == float("inf") else x

    def visit(state, player, depth, node_id, alpha, beta):
        stats["visited_nodes"] += 1
        trace.append({"event": "enter", "id": node_id, "player": player, "depth": depth,
                      "alpha": bound_number(alpha), "beta": bound_number(beta)})
        terminal = len(state) == 2 if kind == "tree" else sum(state) == 0
        cutoff = kind == "tree" and not terminal and depth >= c["depth"]
        if terminal or cutoff:
            if kind == "nim":
                value = -player  # 手番側は合法手がないので負け。
            elif terminal:
                value = c["leaves"]["ABC".index(state[0]) * 2 + int(state[1]) - 1]
            else:
                value = c["heuristics"][0 if state == "R" else "ABC".index(state) + 1]
            category = "terminal" if terminal else "cutoff"
            stats["evaluations"] += 1
            stats[category + "_evaluations"] += 1
            trace.append({"event": "evaluate", "id": node_id, "kind": category, "value": value})
            return value, None

        if kind == "tree":
            children = [(x, x, x) for x in (list("ABC") if state == "R" else [state + "1", state + "2"])]
        else:
            children = [(a, apply_move(state, a), f"{a[0]}:{a[1]}") for a in legal_moves(state)]
        if c["reverse"]:
            children.reverse()
        initial_alpha, initial_beta = alpha, beta
        value = -float("inf") if player == 1 else float("inf")
        best_action = None
        for index, (action, child, token) in enumerate(children):
            score, _ = visit(child, -player, depth + 1, node_id + "/" + token, alpha, beta)
            # 厳密な不等号で、同点時は最初の手を保持する。
            if best_action is None or (score > value if player == 1 else score < value):
                value, best_action = score, action
            if ab:
                if player == 1:
                    alpha = max(alpha, value)
                else:
                    beta = min(beta, value)
            trace.append({"event": "update", "id": node_id, "action": action, "value": value,
                          "alpha": bound_number(alpha), "beta": bound_number(beta)})
            if ab and alpha >= beta:
                skipped = [node_id + "/" + x[2] for x in children[index + 1:]]
                if skipped:
                    pruned.extend(skipped)
                    trace.append({"event": "prune", "id": node_id, "skipped": skipped,
                                  "value": value, "bound": "lower" if player == 1 else "upper"})
                break
        certainty = "upper" if ab and value <= initial_alpha else "lower" if ab and value >= initial_beta else "exact"
        trace.append({"event": "return", "id": node_id, "value": value, "bound": certainty})
        return value, best_action

    value, action = visit("R" if kind == "tree" else c["piles"], c["player"], 0, "R", -float("inf"), float("inf"))
    return {"value": value, "best_action": action, **stats, "pruned": pruned, "trace": trace}


def search_tree(config=None):
    return _search(_config({} if config is None else config, "tree"), "tree")


def search_nim(config=None):
    return _search(_config({} if config is None else config, "nim"), "nim")


def summary(result):
    """基準JSONには値と計数を保存。完全な探索履歴は各result['trace']に残す。"""
    return {key: deepcopy(value) for key, value in result.items() if key != "trace"}


def experiment_configs():
    """六つの独立変更。各ケースの基準をコピーして一項目だけ変える。"""
    cases = {}
    for name in ("leaf", "order", "depth", "heuristic", "piles", "algorithm"):
        kind = "nim" if name == "piles" else "tree"
        base = deepcopy(NIM_BASE if kind == "nim" else TREE_BASE)
        if name == "order":
            base["algorithm"] = "alphabeta"
        if name == "heuristic":
            base["depth"] = 1
        changed = deepcopy(base)
        if name == "leaf": changed["leaves"][2] = 6
        if name == "order": changed["reverse"] = True
        if name == "depth": changed["depth"] = 1
        if name == "heuristic": changed["heuristics"][1] = 5
        if name == "piles": changed["piles"] = [2, 2]
        if name == "algorithm": changed["algorithm"] = "alphabeta"
        cases[name] = {"kind": kind, "base": base, "changed": changed}
    return cases


def demo():
    changes = {}
    for key, case in experiment_configs().items():
        run = search_nim if case["kind"] == "nim" else search_tree
        changes[key] = {"before": summary(run(case["base"])), "after": summary(run(case["changed"]))}
    return {"tree": summary(search_tree()), "alphabeta": summary(search_tree({"algorithm": "alphabeta"})),
            "cutoff": summary(search_tree({"depth": 1})), "nim": summary(search_nim()), "changes": changes}



## 基準を保持

予想：固定木のMAXはA・B・Cのどれを選ぶ？ニムは左から何個取る？想定結果は木3・A、ニム+1・[0,1]。訪問数と評価数も分けて読みます。

In [ ]:
baseline_tree_config = deepcopy(TREE_BASE)
baseline_nim_config = deepcopy(NIM_BASE)
baseline_tree = search_tree(baseline_tree_config)
baseline_nim = search_nim(baseline_nim_config)
baseline_result = demo()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))

## 完全な探索履歴

enterは訪問、evaluateは終端または打切り、updateは探索済み候補からの更新、pruneは未探索の子を省略、returnは上限・下限またはその探索条件の値です。Bの上限2を確定値と呼べない理由を説明しましょう。

In [ ]:
baseline_ab = search_tree({"algorithm": "alphabeta"})
for event in baseline_ab["trace"]:
    print(event)

## 変更：B1の利得2→6

**問い・予想：**Bが選ばれるだろうか。

手順：対応ケースの基準をdeepcopyし、一項目だけ変更した入力で実行。

**想定結果：**値3・A → 値6・B。MINの値も6になる。

**説明：**相手が9でなく6を選ぶのはなぜか。

復帰はカーネル再起動→全セル実行。

In [ ]:
case_leaf = experiment_configs()["leaf"]
trial_leaf_config = deepcopy(case_leaf["changed"])
trial_leaf = search_tree(trial_leaf_config)
print("変更前", baseline_result["changes"]["leaf"]["before"])
print("変更後", summary(trial_leaf))

## 変更：αβの探索順を逆順

**問い・予想：**根値と評価数のどちらが変わるだろうか。

手順：対応ケースの基準をdeepcopyし、一項目だけ変更した入力で実行。

**想定結果：**値3・Aは同じ。終端評価4→6。各子の順序も逆になる。

**説明：**Aを後に調べると省略が減る理由は何か。

復帰はカーネル再起動→全セル実行。

In [ ]:
case_order = experiment_configs()["order"]
trial_order_config = deepcopy(case_order["changed"])
trial_order = search_tree(trial_order_config)
print("変更前", baseline_result["changes"]["order"]["before"])
print("変更後", summary(trial_order))

## 変更：探索深さ2→1

**問い・予想：**深さ1でもAを選ぶだろうか。

手順：対応ケースの基準をdeepcopyし、一項目だけ変更した入力で実行。

**想定結果：**確定した根値3・A → 推定値4・B。終端評価0、打切り評価3。

**説明：**深さ1の4を確定した利得と呼べるか。

復帰はカーネル再起動→全セル実行。

In [ ]:
case_depth = experiment_configs()["depth"]
trial_depth_config = deepcopy(case_depth["changed"])
trial_depth = search_tree(trial_depth_config)
print("変更前", baseline_result["changes"]["depth"]["before"])
print("変更後", summary(trial_depth))

## 変更：深さ1でAの推定値1→5

**問い・予想：**推定値だけで手が変わるだろうか。

手順：対応ケースの基準をdeepcopyし、一項目だけ変更した入力で実行。

**想定結果：**推定値4・B → 推定値5・A。確定利得3とは違う。

**説明：**手が全探索と一致すれば、推定値も正しいか。

復帰はカーネル再起動→全セル実行。

In [ ]:
case_heuristic = experiment_configs()["heuristic"]
trial_heuristic_config = deepcopy(case_heuristic["changed"])
trial_heuristic = search_tree(trial_heuristic_config)
print("変更前", baseline_result["changes"]["heuristic"]["before"])
print("変更後", summary(trial_heuristic))

## 変更：ニム(3,2)→(2,2)

**問い・予想：**同じ数の山は先手に有利だろうか。

手順：対応ケースの基準をdeepcopyし、一項目だけ変更した入力で実行。

**想定結果：**MAX先手の値+1 → −1。訪問86→33、終端37→14。負け局面では同点の最初の手を選ぶ。

**説明：**なぜ値−1でも合法手が返るのか。

復帰はカーネル再起動→全セル実行。

In [ ]:
case_piles = experiment_configs()["piles"]
trial_piles_config = deepcopy(case_piles["changed"])
trial_piles = search_nim(trial_piles_config)
print("変更前", baseline_result["changes"]["piles"]["before"])
print("変更後", summary(trial_piles))

## 変更：ミニマックス→αβ

**問い・予想：**省略によって根値が変わるだろうか。

手順：対応ケースの基準をdeepcopyし、一項目だけ変更した入力で実行。

**想定結果：**値3・Aは同じ。訪問10→8、終端評価6→4。

**説明：**未探索のB2・C2の確定値を表示できない理由は何か。

復帰はカーネル再起動→全セル実行。

In [ ]:
case_algorithm = experiment_configs()["algorithm"]
trial_algorithm_config = deepcopy(case_algorithm["changed"])
trial_algorithm = search_tree(trial_algorithm_config)
print("変更前", baseline_result["changes"]["algorithm"]["before"])
print("変更後", summary(trial_algorithm))

## 基準の保持と復帰

下の検査が成功すれば、基準と6実験が混ざっていません。ValueErrorは入力の範囲と型を、NameErrorは実行順とカーネルを確認します。定義セルを消してしまった場合は保存した元ファイルへ戻します。自分の変更は新しいセルで基準のコピーを使います。

In [ ]:
assert baseline_tree_config == TREE_BASE
assert baseline_nim_config == NIM_BASE
assert baseline_tree == search_tree(TREE_BASE)
assert baseline_nim == search_nim(NIM_BASE)
assert baseline_result == demo()
for key, case in experiment_configs().items():
    run = search_nim if case["kind"] == "nim" else search_tree
    assert globals()["trial_" + key + "_config"] == case["changed"]
    assert globals()["trial_" + key] == run(case["changed"])
print("基準入力・結果を保持しました")